# Pilot runs — slm-audio-evidence (Colab)

Runtime → Change runtime type → **T4 GPU**. Затем — ячейки сверху вниз.

По умолчанию инференс НЕ перезапускается: `responses.jsonl` и ручная разметка `manual-M1` уже закоммичены в `results/`. Ячейки ниже сразу переходят к прогону LLM-судьи и сравнению с ней — аудио для этого не нужно (судья читает только текстовый транскрипт из манифеста).

Нужно пересчитать инференс с нуля (новые аудио/модели/промпты)? Поставь `RUN_INFERENCE = True` в соответствующей ячейке — тогда понадобится `pilot_audio.zip`, загруженный через Files panel (левая панель, drag & drop) в `/content/`.

Нет доступа на чтение/клонирование репозитория? Ниже (ячейка 2) есть флаг `USE_LOCAL_ZIP` — поставь `True` и загрузи zip репозитория через Files panel вместо `git clone`.

In [ ]:
!nvidia-smi -L

In [ ]:
import subprocess

def sh(cmd: str) -> None:
    subprocess.run(cmd, shell=True, check=True)

# False (default): git clone from GitHub (needs read access to the repo).
# True: no repo access -- unzip a manually uploaded copy instead (upload via Files panel, left sidebar).
USE_LOCAL_ZIP = False
REPO_ZIP_PATH = "/content/slm-audio-evidence.zip"

if USE_LOCAL_ZIP:
    sh(f"mkdir -p slm-audio-evidence && unzip -q -o {REPO_ZIP_PATH} -d slm-audio-evidence")
else:
    sh("git clone https://github.com/ladnlav/slm-audio-evidence.git")
%cd slm-audio-evidence

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# True only to regenerate responses.jsonl from scratch (new audio/models/prompts).
# False (default): skip audio + inference entirely -- responses.jsonl and the
# manual-M1 human labels are already committed in results/.
RUN_INFERENCE = False

# Only read if RUN_INFERENCE is True. Upload pilot_audio.zip to /content first
# (drag & drop into the Files panel, left sidebar).
AUDIO_ZIP_PATH = "/content/pilot_audio.zip"

In [ ]:
if RUN_INFERENCE:
    sh(f'unzip -q -o {AUDIO_ZIP_PATH} -d .')
    sh('ls data/audio/spoken_squad_test | head -3')
    import json, os
    rows = [json.loads(l) for l in open('data/manifests/pilot.jsonl', encoding='utf-8')]
    miss = [r['id'] for r in rows if not os.path.exists(r['audio_path'])]
    print(len(rows), 'items,', len(miss), 'missing audio')
    print(miss[:5])
else:
    print('RUN_INFERENCE=False -- skipping audio upload/check (responses.jsonl already in results/).')

In [ ]:
# Run 1 (the headline): Qwen2-Audio, plain prompt. First run also downloads the weights (~15-30 min).
if RUN_INFERENCE:
    sh('python -m src.inference --model qwen2audio --strategy plain --data data/manifests/pilot.jsonl --out results/')
else:
    print('RUN_INFERENCE=False -- skipping.')

In [ ]:
# Run 2: Qwen2-Audio, IDK prompt
if RUN_INFERENCE:
    sh('python -m src.inference --model qwen2audio --strategy s1_idk --data data/manifests/pilot.jsonl --out results/')
else:
    print('RUN_INFERENCE=False -- skipping.')

In [ ]:
# IMPORTANT before runs 3-4: free VRAM from Qwen2-Audio -- Runtime -> Restart runtime,
# then re-run cells 2 (clone/cd), 3 (pip install), 4 (flags) and, if RUN_INFERENCE, cell 5 (unzip)
# -- then continue here.
if RUN_INFERENCE:
    sh('python -m src.inference --model cascade --strategy plain --data data/manifests/pilot.jsonl --out results/')
else:
    print('RUN_INFERENCE=False -- skipping.')

In [ ]:
# Run 4: cascade, IDK prompt
if RUN_INFERENCE:
    sh('python -m src.inference --model cascade --strategy s1_idk --data data/manifests/pilot.jsonl --out results/')
else:
    print('RUN_INFERENCE=False -- skipping.')

In [ ]:
# Zip and download everything produced so far (run after EACH finished run -- do not wait for all four)
if RUN_INFERENCE:
    sh("zip -q -r results_runs.zip results -x '*.gitkeep'")
    from google.colab import files
    files.download('results_runs.zip')
else:
    print('RUN_INFERENCE=False -- nothing new to zip.')

## LLM-судья (категория B) vs ручная разметка manual-M1
`results/<run_id>/responses.jsonl` уже есть (см. выше). Категория B (label=answer) в них размечена людьми вручную (`results/<run_id>/responses_judged.jsonl`, `judge: "manual-M1"`, committed — см. docs/decisions.md) — это уже готовая истина, повторную слепую разметку делать не нужно.

Судья пишет в **отдельную** папку `results/<run_id>/llm_audit/`, а не поверх `responses_judged.jsonl` — тот файл с ручной разметкой нельзя перезаписывать, он невоспроизводим.

Запускаем ОДНИМ Python-процессом (не 4 отдельных вызова) — модель-судья (Qwen2.5-7B-Instruct, int8) грузится один раз и переиспользуется на все 4 прогона: быстрее и без риска VRAM-утечек между процессами, как при повторных инференс-прогонах выше. Если увидите OOM — Runtime → Restart runtime, затем заново ячейки 2 (cd), 3 (pip install), 4 (флаги) и эту.

In [ ]:
from src.judges import build_judge
from src.run_eval import run_evaluation

RUNS = [
    "qwen2audio_plain_20260712",
    "qwen2audio_s1_idk_20260712",
    "cascade_plain_20260712",
    "cascade_s1_idk_20260712",
]

judge = build_judge("local")  # Qwen2.5-7B-Instruct, int8 -- loads once (~2-4 min)
for run_id in RUNS:
    run_evaluation(
        manifest_path="data/manifests/pilot.jsonl",
        responses_path=f"results/{run_id}/responses.jsonl",
        out_dir=f"results/{run_id}/llm_audit",  # separate dir -- never touches committed manual-M1
        judge=judge,
    )

Сравнить свежие вердикты судьи с manual-M1 (гейт согласия ROLE_M3: 80%; если ниже — не публиковать LLM-judge цифры, см. вывод команды):

In [ ]:
!python scripts/audit_judge.py compare --judged "results/*/responses_judged.jsonl"

In [ ]:
# Скачать вердикты судьи + отчёт о согласии
!zip -q -r judge_audit.zip results/*/llm_audit results/judge_audit -x '*.gitkeep'
from google.colab import files
files.download('judge_audit.zip')